# Step 01 — SQLite post-fit pipeline and hidden-mechanism simulation

This notebook validates the implemented **step 01** pipeline against the historical SQLite studies.

Scope of this step:

- read best/top-N trials directly from SQLite without Optuna;
- export normalized effective parameters;
- demonstrate exact `d × pk` structural confounding through `P_gap_eff`;
- simulate representative best trials with hidden-current outputs.

This step remains **provisional**. The historical DBs are single-current fits and do not replace the later six-sweep cell-specific inference.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get('ASTROMODEL_PROJECT_ROOT', Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.optuna_sqlite import read_best_trial
from src.postfit_sqlite import (
    DEFAULT_REPRESENTATIVE_DBS,
    d_pk_invariance_check,
    run_step01_postfit_sqlite,
)

print(f'PROJECT_ROOT={PROJECT_ROOT}')

In [ ]:
results = run_step01_postfit_sqlite(PROJECT_ROOT, top_n=5)
top_trials = results['top_trials_all_dbs']
effective_summary = results['effective_parameter_summary']
representative_summary = results['representative_mechanism_summary']
representative_simulations = results['representative_simulations']

print('written outputs:', sorted((PROJECT_ROOT / 'outputs' / 'postfit_sqlite').glob('*.csv')))
print('top-trial rows:', len(top_trials))

## Direct SQLite post-fit tables

In [ ]:
display(top_trials.head(15))
display(effective_summary)

summary_by_condition = effective_summary.groupby('condition')[['P_gap_eff', 'gamma_t_eff', 'gamma_s_eff', 'volume_ratio_wa_wo']].median()
display(summary_by_condition)

## Exact `d/pk` invariance demonstration

In [ ]:
base_record = read_best_trial(PROJECT_ROOT / 'data' / '1_Initial_xp_fit' / 'CONTROL_75nA.db')
check = d_pk_invariance_check(base_record.params, experiment_type=base_record.condition, current_na=base_record.current_na, scale_factor=3.0)

demo = pd.DataFrame([
    {
        'P_gap_eff_a': check.P_gap_eff_a,
        'P_gap_eff_b': check.P_gap_eff_b,
        'I_kgap_a': check.I_kgap_a,
        'I_kgap_b': check.I_kgap_b,
        'max_abs_dzdt_diff': float(abs(check.dzdt_a - check.dzdt_b).max()),
    }
])
display(demo)

## Representative Vm traces

In [ ]:
plt.figure(figsize=(9, 5))
for db_name, sim in representative_simulations.items():
    plt.plot(sim['t_ms'] / 1000.0, sim['Vm'], label=db_name)
plt.xlabel('Time (s)')
plt.ylabel('Vm (mV)')
plt.title('Representative best-trial Vm traces')
plt.legend()
plt.tight_layout()
plt.show()

## Representative hidden-current overlays

In [ ]:
fig, axes = plt.subplots(len(DEFAULT_REPRESENTATIVE_DBS), 2, figsize=(12, 10), sharex=False)
for row_idx, db_name in enumerate(DEFAULT_REPRESENTATIVE_DBS):
    sim = representative_simulations[db_name]
    t_s = sim['t_ms'] / 1000.0
    axes[row_idx, 0].plot(t_s, sim['currents']['I_Kir'], label='I_Kir')
    axes[row_idx, 0].plot(t_s, sim['currents']['I_kgap'], label='I_kgap')
    axes[row_idx, 0].plot(t_s, sim['currents']['I_leak'], label='I_leak')
    axes[row_idx, 0].set_title(f'{db_name} currents')
    axes[row_idx, 0].set_ylabel('Current (a.u.)')
    axes[row_idx, 0].legend(loc='upper right')

    axes[row_idx, 1].plot(t_s, sim['derived']['K_o'], label='K_o')
    axes[row_idx, 1].plot(t_s, sim['derived']['DK_a'], label='DK_a')
    axes[row_idx, 1].set_title(f'{db_name} derived states')
    axes[row_idx, 1].set_ylabel('Concentration proxy (a.u.)')
    axes[row_idx, 1].legend(loc='upper right')

for ax in axes[-1, :]:
    ax.set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Representative mechanism and proxy summary

In [ ]:
display(representative_summary)

## Interpretation

The historical SQLite studies are now readable without Optuna, structural `d/pk` confounding is demonstrated explicitly, and representative best trials expose hidden-current and proxy summaries. These outputs are useful for debugging and reviewer-facing diagnostics, but they remain provisional because they come from historical single-current fits rather than the final cell-specific six-sweep pipeline.